# 使用scGPT进行基因调控网络（GRN）推断

本教程演示如何使用scGPT模型进行基因调控网络（Gene Regulatory Network, GRN）推断。GRN推断是单细胞数据分析中的重要任务，它可以帮助我们理解基因之间的调控关系。

scGPT通过分析Transformer模型中的注意力权重来推断潜在的调控关系。注意力权重可以被解释为基因之间的关联强度。


In [1]:
# 导入必要的库
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# 添加scGPT路径
sys.path.insert(0, "../")
import scgpt as scg
from scgpt.model import TransformerModel
from scgpt.tokenizer import GeneVocab

print("✅ 所有库加载完成！")

In [2]:
# 设置参数
model_dir = Path("../save/scGPT_human")
data_path = Path("../data/GRN/demo.h5ad")
save_dir = Path("../results/GRN")

# 创建保存目录
save_dir.mkdir(parents=True, exist_ok=True)

print("✅ 参数设置完成！")

In [3]:
# 加载数据
adata = sc.read_h5ad(data_path)

# 预处理
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(f"✅ 数据加载完成！形状: {adata.shape}")

In [4]:
# 加载预训练模型
model = scg.model.TransformerModel.load_pretrained(model_dir)
model.eval()

# 获取设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("✅ 预训练模型加载完成！")

In [5]:
# 获取注意力权重
attention_weights = scg.tasks.infer_grn(
    adata,
    model,
    model_dir,
    batch_size=32,
    device=device,
)

print(f"✅ GRN推断完成！注意力权重形状: {attention_weights.shape}")

In [6]:
# 分析GRN
# 1. 获取基因列表
genes = adata.var.index.tolist()

# 2. 计算每个基因的入度和出度
out_degree = attention_weights.sum(axis=1)
in_degree = attention_weights.sum(axis=0)

# 3. 找出关键调控基因（高入度或高出度）
top_regulators = np.argsort(out_degree)[::-1][:20]
print("Top 20 调控基因:")
for i, idx in enumerate(top_regulators):
    print(f"{i+1}. {genes[idx]} (出度: {out_degree[idx]:.2f})")

# 4. 可视化注意力权重矩阵
plt.figure(figsize=(12, 10))
sns.heatmap(
    attention_weights[:50, :50],
    xticklabels=genes[:50],
    yticklabels=genes[:50],
    cmap="viridis",
    cbar=True
)
plt.title("基因注意力权重矩阵（前50个基因）")
plt.xlabel("目标基因")
plt.ylabel("源基因")
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()

print("✅ GRN分析完成！")

In [7]:
# 保存GRN结果
# 1. 保存注意力权重矩阵
np.save(save_dir / "attention_weights.npy", attention_weights)

# 2. 保存调控关系列表
regulatory_links = []
threshold = 0.01  # 注意力权重阈值

for i in range(len(genes)):
    for j in range(len(genes)):
        if attention_weights[i, j] > threshold:
            regulatory_links.append({
                "source": genes[i],
                "target": genes[j],
                "weight": float(attention_weights[i, j])
            })

links_df = pd.DataFrame(regulatory_links)
links_df.to_csv(save_dir / "regulatory_links.csv", index=False)

print(f"✅ GRN结果已保存！共发现 {len(regulatory_links)} 条调控关系")